# 03 — The SET friction layer

**Phase 3 deliverable:** a buy-and-hold agent backtested through the friction layer, with the
frictionless-vs-friction gap printed.

Upstream's agents buy and sell **one share** at zero cost with unlimited shorting. On SET the
board lot is 100, prices sit on a tick table, commission plus VAT applies both ways, moves are
bounded at ±30%, and retail shorting needs SBL. All three upstream assumptions flatter the result.

> ### Every number in `configs/market.yaml` is reconstructed, not verified
> Spec R13 requires checking the tick table, board-lot exceptions and commission schedule against
> SET's published rules before trusting any backtest figure. That has **not** been done. The tick
> table's shape is stable and widely reproduced; the commission rate varies by broker, channel and
> turnover tier, and the default is a plausible retail internet rate rather than a quoted one.

In [6]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
import pandas as pd
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

## The tick table

The minimum price increment is a step function of price level, so any fill must be snapped to a
valid tick.

In [7]:
from stock_retrofit.market import price_limits, snap_to_tick, tick_size

for price in [1.5, 4.98, 9.95, 24.9, 88.88, 150.4, 275.3, 512.7]:
    print(f"  {price:8.2f}  tick {tick_size(price):5.2f}  "
          f"down {snap_to_tick(price, mode='down'):8.2f}  "
          f"up {snap_to_tick(price, mode='up'):8.2f}")

floor, ceiling = price_limits(150.0)
print(f"\nprevious close 150.00 -> floor {floor}, ceiling {ceiling} (±30%, snapped to tick)")

      1.50  tick  0.01  down     1.50  up     1.50
      4.98  tick  0.02  down     4.98  up     4.98
      9.95  tick  0.05  down     9.95  up     9.95
     24.90  tick  0.10  down    24.90  up    24.90
     88.88  tick  0.25  down    88.75  up    89.00
    150.40  tick  0.50  down   150.00  up   150.50
    275.30  tick  1.00  down   275.00  up   276.00
    512.70  tick  2.00  down   512.00  up   514.00

previous close 150.00 -> floor 105.0, ceiling 195.0 (±30%, snapped to tick)


## What the frictions cost

The same agent, the same bars, two configurations. Spec R11 makes the gap a headline result
rather than a footnote.

In [8]:
from stock_retrofit.config import EvalConfig, MarketConfigSpec
from stock_retrofit.data import load
from stock_retrofit.agents import build, run_agent_walk_forward, agent_results_table, render_agent_table

eval_cfg, market_spec = EvalConfig.load(), MarketConfigSpec.load()
df = load("KBANK")
config = market_spec.build(symbol="KBANK")

print(f"board lot {config.board_lot}, commission {market_spec.commission_rate:.3%} "
      f"+ {market_spec.vat_rate:.0%} VAT, round trip {market_spec.round_trip_cost:.3%}")
print(f"short selling: {'enabled' if config.allow_short else 'DISABLED (no SBL)'}")

result = run_agent_walk_forward(
    build("buy_and_hold", name="buy_and_hold"), df,
    splitter=eval_cfg.splitter(), config=config,
    window=eval_cfg.window(), symbol="KBANK", seed=eval_cfg.seed,
)
print(render_agent_table(agent_results_table([result]), title="KBANK — buy and hold"))

board lot 100, commission 0.157% + 7% VAT, round trip 0.336%
short selling: DISABLED (no SBL)
KBANK — buy and hold
       agent symbol  folds ret_friction ret_frictionless friction_gap sharpe_friction sharpe_frictionless  trades  costs  max_dd
buy_and_hold  KBANK      8       +7.82%           +8.09%       +0.27%           +1.41               +1.45       8 13,313 -11.33%

1 of 1 agents are profitable frictionless; 1 survive SET frictions. Mean cost of frictions: +0.27% per fold.


## Rejections are recorded, not swallowed

An order that cannot be filled — below a board lot, outside the price limit, a short with no SBL,
above the participation cap — produces a `Fill` with a `rejected` reason. Silent rejection would
make an agent look better than it is.

In [9]:
from stock_retrofit.market import MarketConfig, SETMarket

thin = df.tail(300).reset_index(drop=True).copy()
thin["volume"] = 5_000.0                      # a very thin session
market = SETMarket(thin, MarketConfig(initial_cash=50_000_000, participation_cap=0.05))

market.buy(10, fraction=1.0)                  # capped by participation
market.sell(11, shares=999_999)               # more than held, shorting off
market.buy(12, shares=50)                     # below one board lot
for fill in market.fills:
    print(f"  {fill.side:4s} filled {fill.filled:>8,}  rejected={fill.rejected}")

market.assert_invariants()
print("\ninvariants hold: on-tick, on-lot, fees ≥ 0, no fill outside the day's range")

  buy  filled      200  rejected=participation_cap
  sell filled      200  rejected=None
  buy  filled        0  rejected=below_board_lot

invariants hold: on-tick, on-lot, fees ≥ 0, no fill outside the day's range


## BAY carries a participation cap by default

The instrument registry knows BAY is ~72–76% MUFG-held, so a consumer building a backtest can
find out programmatically rather than by remembering.

In [10]:
from stock_retrofit.data import participation_cap_for

for symbol in ["KBANK", "SCB", "BAY"]:
    cap = participation_cap_for(symbol)
    print(f"  {symbol:6s} participation cap: {'none' if cap is None else f'{cap:.0%} of session volume'}")

  KBANK  participation cap: none
  SCB    participation cap: none
  BAY    participation cap: 5% of session volume
